# Experiment 1.3b-i-A: `sigma_1(W)` growth under SGD vs Muon

## Counterpart identity and truthful scope

This notebook is the **paired analysis notebook** for:

- `experiments/Tier_1_Core_Mechanism_Tests/Exp_1.3_Singular_Value_Spectrum_Evolution/1.3b_Spectrum_Gradient_vs_Weight_Spectrum_Correlation/1.3b-i_Feedback_Loop_Does_SGD_Anisotropy_Self-Amplify/1.3b-i-A_Feedback_Measure_sigma1W_Growth_Rate/run_experiment.py`

This second-pass update is intentionally conservative:

- it **keeps the original deterministic deep-linear toy study**,
- it uses the **script's returned results** instead of re-implementing the experiment core,
- it narrows earlier overclaims,
- and it strengthens the fit-comparison reporting with an explicit **T3 diagnostic** plus lightweight **layer-level uncertainty summaries**.

## What this pair now aims to establish

Within a fixed-seed toy setup, compare SGD with momentum against Muon on:

1. growth trajectories of the leading singular value `sigma_1(W)`,
2. final spectral concentration / anisotropy,
3. simple fit-family comparisons for `log(sigma_1(W))`, and
4. limited coupling diagnostics between weights and gradients/updates.

## What this pair does **not** establish by itself

- It is **not** a statistical multi-seed study.
- A scalar correlation between top singular-value magnitudes is **not** direct singular-vector alignment.
- Strong language such as **"exponential self-amplification"** is only warranted if the within-SGD exponential fit actually beats the polynomial alternative in the observed run.
- The uncertainty summaries below are only **across the small set of layers in this one run**; they are descriptive, not seed-level inference.
- For Muon, the **raw gradient**, **orthogonalized gradient**, and **actual momentum update** are different objects and must not be conflated.



## 1. Notebook-safe setup and script import

The notebook avoids `__file__`, avoids forcing an Agg-only backend, and loads the paired script directly from disk. The experiment is run by calling the script's `run_experiment()` function, so there is no duplicated training logic here.


In [ ]:

from pathlib import Path
import importlib.util
import time

import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
except ImportError:
    def display(obj):
        print(obj)

    class Markdown(str):
        pass

try:
    import pandas as pd
except ImportError:
    pd = None

PAIR_RELATIVE = Path('experiments/Tier_1_Core_Mechanism_Tests/Exp_1.3_Singular_Value_Spectrum_Evolution/1.3b_Spectrum_Gradient_vs_Weight_Spectrum_Correlation/1.3b-i_Feedback_Loop_Does_SGD_Anisotropy_Self-Amplify/1.3b-i-A_Feedback_Measure_sigma1W_Growth_Rate')


def locate_pair_dir():
    cwd = Path.cwd().resolve()
    for root in [cwd] + list(cwd.parents):
        candidate = root / PAIR_RELATIVE
        if candidate.exists():
            return root, candidate
    if (cwd / 'run_experiment.py').exists() and (cwd / 'run_experiment.ipynb').exists():
        return cwd, cwd
    raise FileNotFoundError(f'Could not locate pair directory from {cwd}')


def load_script_module(script_path: Path):
    spec = importlib.util.spec_from_file_location('sigma1_growth_rate_pair', script_path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


def to_table(rows):
    if pd is None:
        return rows
    return pd.DataFrame(rows)


REPO_ROOT, PAIR_DIR = locate_pair_dir()
SCRIPT_PATH = PAIR_DIR / 'run_experiment.py'
NOTEBOOK_PATH = PAIR_DIR / 'run_experiment.ipynb'
pair_module = load_script_module(SCRIPT_PATH)

print(f'Repo root:   {REPO_ROOT}')
print(f'Pair dir:    {PAIR_DIR}')
print(f'Script path: {SCRIPT_PATH}')
print(f'Notebook:    {NOTEBOOK_PATH}')
print('Loaded pair module successfully.')



## 2. Run the counterpart experiment and save reusable artifacts

This cell runs exactly the script's reusable experiment function, generates the paired summary figure, and saves structured artifacts for later inspection.


In [ ]:

wall_clock_start = time.time()
results = pair_module.run_experiment(verbose=False)
figure, plot_path = pair_module.make_plots(results, output_dir=PAIR_DIR, filename='sigma1_growth_rate.png', show=False)
artifact_paths = pair_module.save_results_artifacts(results, output_dir=PAIR_DIR, plot_path=plot_path)
results['artifact_paths'] = artifact_paths
wall_clock_elapsed = time.time() - wall_clock_start

print(f"Internal experiment runtime: {results['runtime_seconds']:.2f} s")
print(f"Notebook cell wall time:     {wall_clock_elapsed:.2f} s")
for name, path in artifact_paths.items():
    print(f"{name:12s}: {path}")

if figure is not None:
    display(figure)
    plt.close(figure)



## 3. Reproducibility, runtime, and configuration log

This is the minimum information needed to reproduce the exact run presented below.


In [ ]:

config = results['config']
problem = results['problem']
summary = results['summary']

run_log_rows = [
    {'field': 'counterpart_script', 'value': str(SCRIPT_PATH)},
    {'field': 'counterpart_notebook', 'value': str(NOTEBOOK_PATH)},
    {'field': 'seed', 'value': config['seed']},
    {'field': 'dim', 'value': config['dim']},
    {'field': 'num_layers', 'value': config['num_layers']},
    {'field': 'num_steps', 'value': config['num_steps']},
    {'field': 'batch_size', 'value': config['batch_size']},
    {'field': 'momentum', 'value': config['momentum']},
    {'field': 'ns_iters', 'value': config['ns_iters']},
    {'field': 'chosen_lr_sgd', 'value': results['learning_rates']['SGD']},
    {'field': 'chosen_lr_muon', 'value': results['learning_rates']['Muon']},
    {'field': 'target_condition_number', 'value': problem['target_condition_number']},
    {'field': 'target_fro_norm', 'value': problem['target_fro_norm']},
    {'field': 'input_mean_abs', 'value': problem['input_mean_abs']},
    {'field': 'input_std', 'value': problem['input_std']},
    {'field': 'internal_runtime_seconds', 'value': results['runtime_seconds']},
    {'field': 'notebook_cell_wall_seconds', 'value': wall_clock_elapsed},
]

display(to_table(run_log_rows))



## 4. Implemented metrics versus non-measured claims

The table below is deliberately explicit about scope. This pair now reports both the retained scalar proxy and one small direct directional diagnostic.


In [ ]:

metric_scope_rows = [
    {
        'quantity': 'sigma_1(W), sigma_n(W), condition number, loss',
        'status': 'directly measured each state',
        'interpretation': 'Tracks spectral growth and anisotropy of the weights.'
    },
    {
        'quantity': 'final singular-value spectra',
        'status': 'directly measured at the final state',
        'interpretation': 'Supports final Gini / anisotropy comparisons.'
    },
    {
        'quantity': 'exp vs polynomial fits for log(sigma_1(W))',
        'status': 'fit-family comparison',
        'interpretation': 'Suggests which simple family better summarizes the observed trajectory.'
    },
    {
        'quantity': 'lagged scalar proxy corr(sigma_1(raw grad used for step t), sigma_1(W_t))',
        'status': 'retained proxy only',
        'interpretation': 'Magnitude co-variation proxy; not a vector alignment measure.'
    },
    {
        'quantity': 'same-state corr(sigma_1(raw grad_t), sigma_1(W_t))',
        'status': 'additional proxy only',
        'interpretation': 'Still scalar and still not directional.'
    },
    {
        'quantity': 'top-singular-vector overlap between W and raw gradient / update',
        'status': 'small direct directional diagnostic',
        'interpretation': 'Uses singular vectors directly, but remains a low-dimensional summary.'
    },
    {
        'quantity': 'broad mechanistic proof of self-amplifying exponential feedback',
        'status': 'not automatically established',
        'interpretation': 'Requires the within-SGD fit comparison and careful caveats.'
    },
]

display(to_table(metric_scope_rows))



## 5. `sigma_1(W)` growth trajectories by optimizer

The next figure isolates the core quantity of interest: the leading singular value of each layer weight through training. Faint lines are individual layers; bold lines are the layer mean.


In [ ]:

steps = results['steps']
sgd = results['optimizers']['SGD']
muon = results['optimizers']['Muon']

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
colors = {'SGD': '#1f77b4', 'Muon': '#d62728'}

ax = axes[0]
for layer in range(config['num_layers']):
    ax.plot(steps, sgd['sigma1_W'][:, layer], color=colors['SGD'], alpha=0.25)
    ax.plot(steps, muon['sigma1_W'][:, layer], color=colors['Muon'], alpha=0.25, linestyle='--')
ax.plot(steps, np.mean(sgd['sigma1_W'], axis=1), color=colors['SGD'], linewidth=3, label='SGD mean')
ax.plot(steps, np.mean(muon['sigma1_W'], axis=1), color=colors['Muon'], linewidth=3, linestyle='--', label='Muon mean')
ax.set_title('sigma_1(W)')
ax.set_xlabel('Step')
ax.set_ylabel('sigma_1(W)')
ax.grid(True, alpha=0.3)
ax.legend()

ax = axes[1]
for layer in range(config['num_layers']):
    ax.plot(steps, np.log(sgd['sigma1_W'][:, layer] + 1e-30), color=colors['SGD'], alpha=0.25)
    ax.plot(steps, np.log(muon['sigma1_W'][:, layer] + 1e-30), color=colors['Muon'], alpha=0.25, linestyle='--')
ax.plot(steps, np.mean(np.log(sgd['sigma1_W'] + 1e-30), axis=1), color=colors['SGD'], linewidth=3, label='SGD mean')
ax.plot(steps, np.mean(np.log(muon['sigma1_W'] + 1e-30), axis=1), color=colors['Muon'], linewidth=3, linestyle='--', label='Muon mean')
ax.set_title('log(sigma_1(W))')
ax.set_xlabel('Step')
ax.set_ylabel('log(sigma_1(W))')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()



### Interpretation

These curves tell us whether SGD and Muon produce visibly different leading-singular-value growth rates in this fixed toy setup. They do **not** by themselves prove an exponential feedback law; that claim depends on the fit comparison below.



## 6. Final spectrum summary view

The original study focused strongly on `sigma_1(W)`. To keep the broader spectral picture honest, we also summarize the final singular-value spectra and their Gini coefficients.


In [ ]:

gini_rows = []
for layer in range(config['num_layers']):
    gini_rows.append({
        'layer': layer,
        'sgd_gini': summary['optimizer_summaries']['SGD']['gini_by_layer'][layer],
        'muon_gini': summary['optimizer_summaries']['Muon']['gini_by_layer'][layer],
        'sgd_minus_muon': (
            summary['optimizer_summaries']['SGD']['gini_by_layer'][layer]
            - summary['optimizer_summaries']['Muon']['gini_by_layer'][layer]
        ),
    })

display(to_table(gini_rows))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
sv_idx = np.arange(1, config['dim'] + 1)

ax = axes[0]
ax.plot(sv_idx, np.mean(sgd['final_sv_spectrum'], axis=0), marker='o', markersize=3, linewidth=2.5, color=colors['SGD'], label='SGD mean final spectrum')
ax.plot(sv_idx, np.mean(muon['final_sv_spectrum'], axis=0), marker='s', markersize=3, linewidth=2.5, linestyle='--', color=colors['Muon'], label='Muon mean final spectrum')
ax.set_title('Mean final singular-value spectrum across layers')
ax.set_xlabel('Singular value index')
ax.set_ylabel('Singular value')
ax.grid(True, alpha=0.3)
ax.legend()

ax = axes[1]
width = 0.35
layer0_x = np.arange(config['dim'])
ax.bar(layer0_x - width / 2, sgd['final_sv_spectrum'][0], width, color=colors['SGD'], alpha=0.65, label='SGD layer 0')
ax.bar(layer0_x + width / 2, muon['final_sv_spectrum'][0], width, color=colors['Muon'], alpha=0.65, label='Muon layer 0')
ax.set_title('Final singular-value spectrum, layer 0')
ax.set_xlabel('Singular value index')
ax.set_ylabel('Singular value')
ax.grid(True, alpha=0.3, axis='y')
ax.legend()

plt.tight_layout()
plt.show()



### Interpretation

The Gini comparison addresses a simpler question than the fit analysis: which optimizer ends with a more unequal singular-value spectrum? This supports statements about **relative anisotropy**, not by itself about the exact growth law.


## 7. Fit-comparison summary and calibrated verdict components

Earlier versions of this pair overclaimed here. This section now separates:

1. per-layer fit details,
2. side-by-side fit-family summaries with lightweight layer-level uncertainty,
3. the explicit **T3 honesty check** that controls whether stronger exponential wording is defensible, and
4. a direct view of whether the SGD layerwise evidence is uniformly against the exponential story in this run.


In [ ]:
fit_rows = []
for optimizer_name in ['SGD', 'Muon']:
    opt_summary = summary['optimizer_summaries'][optimizer_name]
    for row in opt_summary['fit_by_layer']:
        fit_rows.append({
            'optimizer': optimizer_name,
            'layer': row['layer'],
            'exp_a': row['exp_a'],
            'exp_r2': row['exp_r2'],
            'poly_a': row['poly_a'],
            'poly_r2': row['poly_r2'],
            'r2_gap_exp_minus_poly': row['r2_gap_exp_minus_poly'],
            'best_fit': row['best_fit'],
        })

fit_mean_rows = []
for optimizer_name in ['SGD', 'Muon']:
    opt_summary = summary['optimizer_summaries'][optimizer_name]
    fit_means = opt_summary['fit_means']
    fit_uncertainty = opt_summary['fit_uncertainty']
    fit_mean_rows.append({
        'optimizer': optimizer_name,
        'exp_r2_mean': fit_means['exp_r2_mean'],
        'exp_r2_ci95_low': fit_uncertainty['exp_r2']['ci95_low'],
        'exp_r2_ci95_high': fit_uncertainty['exp_r2']['ci95_high'],
        'poly_r2_mean': fit_means['poly_r2_mean'],
        'poly_r2_ci95_low': fit_uncertainty['poly_r2']['ci95_low'],
        'poly_r2_ci95_high': fit_uncertainty['poly_r2']['ci95_high'],
        'r2_gap_exp_minus_poly_mean': fit_means['r2_gap_exp_minus_poly_mean'],
        'r2_gap_ci95_low': fit_uncertainty['r2_gap_exp_minus_poly']['ci95_low'],
        'r2_gap_ci95_high': fit_uncertainty['r2_gap_exp_minus_poly']['ci95_high'],
        'layers_preferring_exponential': fit_uncertainty['layers_preferring_exponential'],
        'layers_preferring_polynomial': fit_uncertainty['layers_preferring_polynomial'],
        'best_fit_by_mean': fit_means['best_fit_by_mean'],
    })

t3 = summary['t3_diagnostic']
t3_rows = [{
    'component': 'T3_sgd_exp_r2_gt_sgd_poly_r2',
    'pass': t3['pass'],
    'mean_r2_gap_exp_minus_poly': t3['mean_r2_gap_exp_minus_poly'],
    'ci95_low': t3['layerwise_r2_gap_summary']['ci95_low'],
    'ci95_high': t3['layerwise_r2_gap_summary']['ci95_high'],
    'layers_preferring_exponential': t3['layer_vote_counts']['layers_preferring_exponential'],
    'layers_preferring_polynomial': t3['layer_vote_counts']['layers_preferring_polynomial'],
    'direction': t3['mean_r2_gap_direction'],
}]

layer_gap_rows = []
for layer_idx, gap in enumerate(t3['layerwise_r2_gap_values']):
    layer_gap_rows.append({
        'layer': layer_idx,
        'sgd_r2_gap_exp_minus_poly': gap,
        'supports_exponential_over_polynomial': gap > 0,
    })

verdict_rows = []
for name, info in summary['verdict_components'].items():
    row = {'component': name, 'pass': info['pass'], 'description': info['description']}
    for key, value in info.items():
        if key not in {'pass', 'description'}:
            row[key] = value
    verdict_rows.append(row)

print('Per-layer fit details:')
display(to_table(fit_rows))
print('Fit-family summary with descriptive layer-level uncertainty:')
display(to_table(fit_mean_rows))
print('Explicit T3 diagnostic:')
display(to_table(t3_rows))
print('SGD layerwise T3 gaps (all layers matter because T3 is the honesty check):')
display(to_table(layer_gap_rows))
print('Verdict components:')
display(to_table(verdict_rows))

fig, axes = plt.subplots(1, 3, figsize=(17.0, 4.8))
optimizers = ['SGD', 'Muon']
x = np.arange(len(optimizers))
width = 0.34

exp_means = [summary['optimizer_summaries'][name]['fit_means']['exp_r2_mean'] for name in optimizers]
poly_means = [summary['optimizer_summaries'][name]['fit_means']['poly_r2_mean'] for name in optimizers]
exp_err = [
    summary['optimizer_summaries'][name]['fit_uncertainty']['exp_r2']['ci95_high']
    - summary['optimizer_summaries'][name]['fit_uncertainty']['exp_r2']['mean']
    for name in optimizers
]
poly_err = [
    summary['optimizer_summaries'][name]['fit_uncertainty']['poly_r2']['ci95_high']
    - summary['optimizer_summaries'][name]['fit_uncertainty']['poly_r2']['mean']
    for name in optimizers
]

ax = axes[0]
ax.bar(x - width / 2, exp_means, width, yerr=exp_err, capsize=5, color='#4c78a8', alpha=0.8, label='Exponential fit')
ax.bar(x + width / 2, poly_means, width, yerr=poly_err, capsize=5, color='#f58518', alpha=0.8, label='Polynomial fit')
ax.set_xticks(x)
ax.set_xticklabels(optimizers)
ax.set_ylabel('Mean layerwise $R^2$')
ax.set_title('Fit-family comparison by optimizer')
ax.legend()
ax.grid(True, alpha=0.25, axis='y')

gap_means = [summary['optimizer_summaries'][name]['fit_means']['r2_gap_exp_minus_poly_mean'] for name in optimizers]
gap_err = [
    summary['optimizer_summaries'][name]['fit_uncertainty']['r2_gap_exp_minus_poly']['ci95_high']
    - summary['optimizer_summaries'][name]['fit_uncertainty']['r2_gap_exp_minus_poly']['mean']
    for name in optimizers
]
ax = axes[1]
bar_colors = ['#1f77b4', '#d62728']
ax.bar(x, gap_means, yerr=gap_err, capsize=5, color=bar_colors, alpha=0.8)
ax.axhline(0.0, color='gray', linestyle=':', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(optimizers)
ax.set_ylabel('Mean $R^2_{exp} - R^2_{poly}$')
ax.set_title('Key T3 diagnostic: is exponential actually better?')
ax.grid(True, alpha=0.25, axis='y')

ax = axes[2]
sgd_layer_gaps = np.asarray(t3['layerwise_r2_gap_values'], dtype=float)
ax.axhline(0.0, color='gray', linestyle=':', alpha=0.7)
ax.bar(np.arange(len(sgd_layer_gaps)), sgd_layer_gaps, color='#1f77b4', alpha=0.85)
ax.set_xticks(np.arange(len(sgd_layer_gaps)))
ax.set_xticklabels([f'Layer {i}' for i in range(len(sgd_layer_gaps))], rotation=20, ha='right')
ax.set_ylabel('SGD $R^2_{exp} - R^2_{poly}$')
ax.set_title('Layerwise SGD T3 gaps')
ax.grid(True, alpha=0.25, axis='y')

plt.tight_layout()
plt.show()


### Interpretation

The most important honesty check is the within-SGD comparison:

- if `exp_R^2(SGD) <= poly_R^2(SGD)`, then this run does **not** justify saying that SGD growth is itself well-described as exponential,
- even if SGD still grows faster or becomes more anisotropic than Muon.

The strengthened **T3 diagnostic** now makes that check explicit in four ways:

1. the mean gap `R^2_exp - R^2_poly`,
2. a descriptive layer-level 95% interval for that gap,
3. the count of layers preferring exponential versus polynomial fits, and
4. the actual layerwise gap values themselves.

In the current default run, the stronger exponential story is especially weak because the layerwise evidence is not merely mixed: the stored T3 diagnostic reports **0 of 4 layers preferring the exponential family**, and the descriptive layer-level interval for the SGD gap remains **entirely below zero**. That still does not create seed-level certainty, but it is a strong reason to keep the final wording calibrated.

These layer-level intervals are useful for transparency, but they are **not** a substitute for multi-seed uncertainty. They only summarize variability across the small number of layers in this one deterministic run.



## 8. Scalar proxy versus direct directional diagnostic

The pair now reports both a retained scalar proxy and one small direct alignment diagnostic. They answer different questions. This section also makes explicit how the **raw-gradient** overlap and **actual-update** overlap differ, since Muon transforms the gradient before the momentum update is formed.


In [ ]:
alignment_diag = summary['alignment_diagnostic']

proxy_rows = []
for optimizer_name in ['SGD', 'Muon']:
    opt_summary = summary['optimizer_summaries'][optimizer_name]
    proxy_rows.append({
        'optimizer': optimizer_name,
        'lagged_scalar_proxy_mean': opt_summary['lagged_corr_proxy_mean'],
        'same_state_raw_grad_corr_mean': opt_summary['same_state_raw_grad_corr_mean'],
        'weight_raw_grad_top_vector_overlap_mean': opt_summary['weight_raw_grad_alignment_mean'],
        'weight_update_top_vector_overlap_mean': opt_summary['weight_update_alignment_mean'],
        'update_minus_raw_overlap_mean': opt_summary['weight_update_minus_raw_alignment_mean'],
    })

alignment_rows = []
for optimizer_name in ['SGD', 'Muon']:
    opt_summary = summary['optimizer_summaries'][optimizer_name]
    align_unc = opt_summary['alignment_uncertainty']
    alignment_rows.append({
        'optimizer': optimizer_name,
        'raw_overlap_mean': opt_summary['weight_raw_grad_alignment_mean'],
        'raw_overlap_ci95_low': align_unc['weight_raw_grad_alignment']['ci95_low'],
        'raw_overlap_ci95_high': align_unc['weight_raw_grad_alignment']['ci95_high'],
        'update_overlap_mean': opt_summary['weight_update_alignment_mean'],
        'update_overlap_ci95_low': align_unc['weight_update_alignment']['ci95_low'],
        'update_overlap_ci95_high': align_unc['weight_update_alignment']['ci95_high'],
        'update_minus_raw_mean': opt_summary['weight_update_minus_raw_alignment_mean'],
        'update_minus_raw_ci95_low': align_unc['weight_update_minus_raw_alignment']['ci95_low'],
        'update_minus_raw_ci95_high': align_unc['weight_update_minus_raw_alignment']['ci95_high'],
    })

display(to_table(proxy_rows))
print('Direct alignment summary with descriptive layer-level intervals:')
display(to_table(alignment_rows))
print('Alignment comparison note:')
display(to_table([alignment_diag['comparison']]))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6))
labels = ['lagged scalar proxy', 'same-state raw-grad corr', 'W/raw-grad overlap', 'W/update overlap']
sgd_vals = [
    summary['optimizer_summaries']['SGD']['lagged_corr_proxy_mean'],
    summary['optimizer_summaries']['SGD']['same_state_raw_grad_corr_mean'],
    summary['optimizer_summaries']['SGD']['weight_raw_grad_alignment_mean'],
    summary['optimizer_summaries']['SGD']['weight_update_alignment_mean'],
]
muon_vals = [
    summary['optimizer_summaries']['Muon']['lagged_corr_proxy_mean'],
    summary['optimizer_summaries']['Muon']['same_state_raw_grad_corr_mean'],
    summary['optimizer_summaries']['Muon']['weight_raw_grad_alignment_mean'],
    summary['optimizer_summaries']['Muon']['weight_update_alignment_mean'],
]

x = np.arange(len(labels))
width = 0.36
ax = axes[0]
ax.bar(x - width / 2, sgd_vals, width, color=colors['SGD'], alpha=0.75, label='SGD')
ax.bar(x + width / 2, muon_vals, width, color=colors['Muon'], alpha=0.75, label='Muon')
ax.axhline(0.0, color='gray', linestyle=':', alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha='right')
ax.set_ylabel('Mean summary value')
ax.set_title('Proxy quantities versus direct top-vector overlap diagnostic')
ax.legend()
ax.grid(True, alpha=0.25, axis='y')

ax = axes[1]
delta_labels = ['SGD', 'Muon']
delta_means = [
    summary['optimizer_summaries']['SGD']['weight_update_minus_raw_alignment_mean'],
    summary['optimizer_summaries']['Muon']['weight_update_minus_raw_alignment_mean'],
]
delta_err = [
    summary['optimizer_summaries'][name]['alignment_uncertainty']['weight_update_minus_raw_alignment']['ci95_high']
    - summary['optimizer_summaries'][name]['alignment_uncertainty']['weight_update_minus_raw_alignment']['mean']
    for name in ['SGD', 'Muon']
]
ax.bar(np.arange(2), delta_means, yerr=delta_err, capsize=5, color=[colors['SGD'], colors['Muon']], alpha=0.8)
ax.axhline(0.0, color='gray', linestyle=':', alpha=0.6)
ax.set_xticks(np.arange(2))
ax.set_xticklabels(delta_labels)
ax.set_ylabel('Mean W/update overlap - W/raw-grad overlap')
ax.set_title('How much the actual update direction differs from the raw gradient')
ax.grid(True, alpha=0.25, axis='y')

plt.tight_layout()
plt.show()


### What the proxy does and does not mean

- `corr(sigma_1(gradient), sigma_1(W))` measures only whether **top singular-value magnitudes** co-vary.
- It does **not** tell us whether the corresponding singular vectors point in the same directions.
- The top-vector overlap diagnostic is therefore more direct, although still far from a complete mechanistic analysis.
- The raw-gradient overlap and update overlap should also be kept separate: the latter is the more faithful diagnostic for the actual parameter step.

In the current default run, the scalar proxy advantage for SGD does **not** line up with a direct W/update overlap advantage. That mismatch is exactly why this notebook treats the scalar correlation as a **proxy with limitations**, not as a proof of directional feedback.



## 9. Final calibrated conclusion

The conclusion below is assembled from the actual returned results rather than from fixed narrative text. The wording is intentionally constrained by the T3 honesty check and by the distinction between scalar proxies and direct alignment diagnostics.


In [ ]:
assessment = summary['assessment']
sgd_summary = summary['optimizer_summaries']['SGD']
muon_summary = summary['optimizer_summaries']['Muon']
t3 = summary['t3_diagnostic']

lines = [
    '### Conclusion for this executed run',
    '',
    f"**Overall assessment:** {assessment['overall_label']}.",
    '',
    '**Observed headline numbers**',
    f"- Chosen learning rates: SGD = {results['learning_rates']['SGD']}, Muon = {results['learning_rates']['Muon']}",
    f"- Final loss: SGD = {sgd_summary['loss_final']:.6e}, Muon = {muon_summary['loss_final']:.6e}",
    f"- Mean exponential-fit slope: SGD = {sgd_summary['fit_means']['exp_slope_mean']:.6f}, Muon = {muon_summary['fit_means']['exp_slope_mean']:.6f}",
    f"- Mean final Gini: SGD = {sgd_summary['gini_mean']:.4f}, Muon = {muon_summary['gini_mean']:.4f}",
    f"- Mean lagged scalar proxy: SGD = {sgd_summary['lagged_corr_proxy_mean']:.4f}, Muon = {muon_summary['lagged_corr_proxy_mean']:.4f}",
    f"- Mean direct W/raw-grad top-vector overlap: SGD = {sgd_summary['weight_raw_grad_alignment_mean']:.4f}, Muon = {muon_summary['weight_raw_grad_alignment_mean']:.4f}",
    f"- Mean direct W/update top-vector overlap: SGD = {sgd_summary['weight_update_alignment_mean']:.4f}, Muon = {muon_summary['weight_update_alignment_mean']:.4f}",
    '',
    '**Key T3 honesty check**',
    f"- T3 status: {'PASS' if t3['pass'] else 'FAIL'}",
    f"- Mean $R^2_{{exp}} - R^2_{{poly}}$ within SGD = {t3['mean_r2_gap_exp_minus_poly']:.4f}",
    f"- Descriptive layer-level 95% interval for that gap: [{t3['layerwise_r2_gap_summary']['ci95_low']:.4f}, {t3['layerwise_r2_gap_summary']['ci95_high']:.4f}]",
    f"- Layer votes: exponential = {t3['layer_vote_counts']['layers_preferring_exponential']}, polynomial = {t3['layer_vote_counts']['layers_preferring_polynomial']}",
    f"- Layerwise SGD gaps: {', '.join(f'{value:.4f}' for value in t3['layerwise_r2_gap_values'])}",
    '',
    '**Interpretation**',
    '- The pair supports statements about **relative growth speed**, **final anisotropy**, and **limited coupling diagnostics** in this deterministic toy setup.',
    '- The pair supports the stronger story of **exponential self-amplification** only if the within-SGD exponential fit beats the polynomial alternative.',
    f"- For this executed run, that stronger story is reported as **{'supported' if assessment['supports_strong_exponential_self_amplification_story'] else 'not supported'}**.",
    f"- Concretely, T3 is **{'passing' if t3['pass'] else 'failing'}**, so the current run {'does' if t3['pass'] else 'does not'} support the stronger exponential wording.",
    f"- In this default run, the T3 evidence is {'uniformly unfavorable across layers' if (t3['layer_vote_counts']['layers_preferring_exponential'] == 0 and t3['layerwise_r2_gap_summary']['ci95_high'] < 0) else 'not uniformly one-sided across layers'}, which is why the conclusion stays conservative.",
    '- The scalar proxy and the direct alignment diagnostic should not be conflated: a scalar-proxy advantage for SGD is not the same as a directional-overlap advantage for the actual update.',
    '',
    '**Caveats carried forward**',
]
for note in assessment['notes']:
    lines.append(f'- {note}')
lines.extend([
    '- The layer-level intervals above are descriptive within this one run, not seed-level uncertainty estimates.',
    '',
    '**Recommended next verification step**',
    '- If a stronger mechanistic claim is desired, the next serious step is a small multi-seed sweep using the same returned metrics, rather than a narrative rewrite alone.',
])

display(Markdown("\n".join(lines)))
